# Long-Term Memory(LTM)
## Implementation

### Problem Statement — Memory-Aware Chatbot

Build a simple **LangGraph chatbot with Long-Term Memory** using `InMemoryStore`.

**Flow:**

```text
    START
      ↓
    Memory Extractor
      ↓
    Memory Retriever
      ↓
    Memory-Aware Chat
      ↓
    END
```

**Nodes:**

1. **`memory_extractor`**

   * Analyze the user's message.
   * Extract useful information.
   * Store important memories in `InMemoryStore`.

2. **`memory_retriever`**

   * Search `InMemoryStore` for memories relevant to the current user message.
   * Return the relevant memories to the graph state.

3. **`memory_aware_chat`**

   * Use the user's message + retrieved memories.
   * Generate a personalized response.

**Example:**

```text
Thread 1:
User: "I prefer Python for backend development."
→ memory_extractor
→ Store: "User prefers Python for backend development."

Thread 2:
User: "Which language should I use for backend?"
→ memory_retriever
→ Retrieves: "User prefers Python..."
→ memory_aware_chat
→ "You prefer Python for backend development."
```

**Constraints:** Use `InMemoryStore`, support different `thread_id`s, and use **semantic search** for memory retrieval.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import uuid

# Long-Term Memory (LTM) imports
# BaseStore: Abstract interface for storing/retrieving memories across conversation sessions
# InMemoryStore: Default implementation that persists memories within a single process lifetime
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langchain_core.runnables import RunnableConfig

In [ ]:
# State definition for LTM-aware chatbot
# - messages: conversation history for context
# - memories: retrieved long-term memories that persist across different conversation sessions/threads
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    memories: list[str]

In [ ]:
# Structured output model for LTM extraction
# Using Pydantic ensures LLM returns parseable, consistent memory items
# This enables reliable storage and retrieval from InMemoryStore
class ExtractPreferencesModel(BaseModel):
    preferences: list[str] = Field(default=[], description="preferences/details which can consider for long term memory")

In [5]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    temperature=0.6
)

In [6]:
# MEMORY_EXTRACTOR_PROMPT

MEMORY_EXTRACTOR_PROMPT = """You are a memory extraction agent.
Analyze the user's message and identify information that would be useful to remember for future conversations.
Store only meaningful, user-specific information such as:
- Preferences
- Skills
- Interests
- Goals
- Ongoing projects
- Important facts

Do not store temporary or irrelevant information.
Return the memories as short, clear statements."""



In [7]:
# MEMORY_AWARE_CHATBOT_PROMPT

MEMORY_AWARE_CHATBOT_PROMPT = """You are a helpful AI assistant.

Answer the user's message naturally.

Use the retrieved long-term memories when they are relevant to the user's request.
Do not mention the memory system or tell the user that you retrieved memories.

Existing Preferences:
{preferences}

Use these preferences to personalize your response when relevant.

If no relevant memories are available, answer using the current conversation context and your general knowledge.

Do not invent information that is not available."""

In [ ]:
def memory_extractor(state: State, config: RunnableConfig, *, store: BaseStore):
    # Hierarchical namespace for LTM isolation
    # ("users", user_id, "details") - enables multi-user LTM segregation
    # Each user has isolated memory namespace, preventing cross-user memory leakage
    namespace = ("users", config["configurable"]["user_id"], "details")

    print("state: ", state)

    current_message = state["messages"][-1]

    # Use structured output to extract LTM-relevant information consistently
    structured_llm = llm.with_structured_output(ExtractPreferencesModel)

    resp = structured_llm.invoke([SystemMessage(content=MEMORY_EXTRACTOR_PROMPT), current_message])

    # Store each extracted preference as a separate memory entry
    # UUID key ensures unique storage; this can cause duplicates (addressed in later notebooks)
    for pref in resp.preferences:
        memory_id = str(uuid.uuid4())
        store.put(namespace=namespace, key=memory_id, value={"data": pref})
        
    return {}

In [ ]:
def memory_retriver(state: State, config: RunnableConfig, *, store: BaseStore):
    # Retrieve all memories from user's namespace
    # store.search() retrieves all stored memories for the current user
    # This enables the chatbot to access previously learned preferences across sessions
    namespace = ("users", config["configurable"]["user_id"], "details")

    memories = store.search(namespace)

    memories_list = []

    # Extract memory data from store items
    for memory in memories:
        memories_list.append(memory.value["data"])

    # Return extracted memories to state for use in chat node
    return {
        "memories" : memories_list
    }

In [ ]:
def chat_node(state: State, config: RunnableConfig, store: BaseStore):
    messages = state["messages"]
    
    # Retrieved LTM memories are injected into the system prompt
    # This makes the LLM aware of user preferences/details from previous sessions
    # The LLM uses these memories to personalize responses without explicit memory references
    memories = state["memories"]

    prompt = MEMORY_AWARE_CHATBOT_PROMPT.format(preferences=memories)

    resp = llm.invoke([SystemMessage(content=prompt)] + messages)

    return {
        "messages": [resp]
    }

In [ ]:
# LTM workflow: Extract → Retrieve → Personalized Response
# 1. memory_extractor: Parses current message for LTM-worthy information and stores it
# 2. memory_retriver: Fetches all user memories from previous conversations
# 3. chat_node: Uses memories to generate context-aware, personalized responses
builder = StateGraph(State)\
    .add_node("memory_extractor", memory_extractor)\
    .add_node("memory_retriver", memory_retriver)\
    .add_node("chat_node", chat_node)\
    .add_edge(START, "memory_extractor")\
    .add_edge("memory_extractor", "memory_retriver")\
    .add_edge("memory_retriver", "chat_node")\
    .add_edge("chat_node", END)

In [ ]:
# Pass InMemoryStore to graph compilation
# The store is injected into node functions via RunnableConfig, enabling LTM access
# This allows memory_extractor to write and memory_retriever to read from persistent storage
store = InMemoryStore()

graph = builder.compile(store=store)

In [13]:
# graph

In [ ]:
# Graph config with user_id for LTM isolation
# user_id is the key to multi-user LTM: each user gets separate memory namespace
# Different user_ids prevent memory cross-contamination in multi-user systems
config = {
    "configurable": {
        "user_id": "user-1"
    }
}

state:  {'messages': [HumanMessage(content='what is lanbda function in python', additional_kwargs={}, response_metadata={}, id='a02fd666-4a83-473f-9760-68db5e9ad606')]}


content=[{'type': 'text', 'text': 'Hi Bhavin! It’s great to see you diving deeper into Python. Since you’re currently exploring lambda functions, it’s a perfect time to break them down.\n\nIn Python, a **lambda function** is a small, anonymous function—meaning it’s a function defined without a name. While a standard function is defined using the `def` keyword, a lambda function is defined using the `lambda` keyword.\n\n### The Syntax\nThe syntax is very concise:\n`lambda arguments: expression`\n\n*   **Arguments:** You can have as many arguments as you want.\n*   **Expression:** There is only one expression, which is evaluated and automatically returned.\n\n### A Simple Comparison\nIf you wanted to write a function that doubles a number, you would normally do this:\n\n```python\ndef double(x):\n    return x * 2\n\nprint(double(5))  # Output: 10\n```\n\nUsing a lambda function, you can write the exact same logic in a single line:\n\n```python\ndouble = lambda x: x * 2\n\nprint(double(5)

In [ ]:
# Test 1: First interaction - extracting and storing LTM
# This message provides context about the user that should be extracted and stored
msg = "my name is bhavin"

resp = graph.invoke({"messages": [HumanMessage(content=msg)]}, config)

print(resp["messages"][-1])

In [ ]:
# Test 2: Subsequent interaction - LTM enables personalized response
# Even though this question doesn't mention the user's name, LTM retrieves "my name is bhavin"
# The chatbot can personalize response using stored memory from previous conversation
msg = "what is lanbda function in python"

resp = graph.invoke({"messages": [HumanMessage(content=msg)]}, config)

print(resp["messages"][-1])

### Issue in this workflow

hit below tab to generate the issue

In [ ]:
# Test 3: again asking the same question 
# here it will again add the prefernce about the python and lambda function in store 
msg = "what is lanbda function in python"

resp = graph.invoke({"messages": [HumanMessage(content=msg)]}, config)

print(resp["messages"][-1])

state:  {'messages': [HumanMessage(content='what is lanbda function in python', additional_kwargs={}, response_metadata={}, id='b57a359c-4cc6-4af9-b5b3-e4877431301d')]}
content=[{'type': 'text', 'text': "Hi Bhavin! It's great to see you diving deeper into Python. Since you're currently exploring lambda functions, I’d be happy to break them down for you.\n\nIn Python, a **lambda function** is essentially a small, anonymous function—meaning it’s a function defined without a name. \n\n### The Basic Syntax\nWhile a standard function is defined using the `def` keyword, a lambda function uses the `lambda` keyword. The structure looks like this:\n\n```python\nlambda arguments: expression\n```\n\n*   **`arguments`**: You can have any number of arguments, but they are separated by commas.\n*   **`expression`**: This is a single expression that gets evaluated and returned. You don't need a `return` statement; it's implicit.\n\n### A Quick Comparison\nLet’s look at a simple function that adds 10 

### Verification

In [ ]:
# Inspect stored LTM: Verify that memories were extracted and stored
# This demonstrates LTM persistence - all extracted memories remain in InMemoryStore
namespace = ("users", config["configurable"]["user_id"], "details")

store.search(namespace)

[Item(namespace=['users', 'user-1', 'details'], key='1ed86c0d-e7e4-4956-84fa-bf75b6564173', value={'data': "The user's name is Bhavin"}, created_at='2026-08-22T15:26:06.641451+00:00', updated_at='2026-08-22T15:26:06.641453+00:00', score=None),
 Item(namespace=['users', 'user-1', 'details'], key='20355a7b-87dd-4cad-8f01-2a8bbcd13027', value={'data': 'Interested in learning Python programming concepts'}, created_at='2026-08-22T15:29:00.723631+00:00', updated_at='2026-08-22T15:29:00.723636+00:00', score=None),
 Item(namespace=['users', 'user-1', 'details'], key='4098c695-47e8-4c95-b356-4a46da554afa', value={'data': 'Currently exploring lambda functions'}, created_at='2026-08-22T15:29:00.723740+00:00', updated_at='2026-08-22T15:29:00.723742+00:00', score=None),
 Item(namespace=['users', 'user-1', 'details'], key='65793b90-37a8-48e1-a772-6d3ad56c41ad', value={'data': 'Interested in learning Python programming'}, created_at='2026-08-22T15:30:22.237566+00:00', updated_at='2026-08-22T15:30:22.

### Problem: Duplicate Long-Term Memories

When the **same user message is processed multiple times**, the memory extraction node creates and stores the same memory again with a new key.

For example:

```text
User: "I prefer Python"

Store:
1 → "User prefers Python"
2 → "User prefers Python"
3 → "User prefers Python"
```

This leads to **duplicate memories and unnecessary storage**.

The system needs a **deduplication or update mechanism** to detect whether a similar memory already exists before creating a new entry.

### Solutions
1. **Deterministic Key / Upsert** — Use a stable key so repeated memories overwrite the same entry.
2. **Exact Duplicate Check** — Compare the new memory with existing memories and skip exact matches.
3. **Semantic Similarity Check** — Use embeddings to detect memories with similar meaning.
4. **LLM-Based Deduplication** — Ask an LLM whether a new memory is duplicate, new, or an update.
5. **Hybrid Deduplication** — Combine exact matching, semantic search, and LLM reasoning for better accuracy.
